# Task-specific Hugging Face models with LangChain
## 22 different models · 22 tasks · 2 examples per task

Each section loads **its own model**, runs two examples, then releases the model from RAM. No shared chat model is used to imitate these tasks.

**Execution mode:** local inference on CPU. Your API token is read from `.env` and authenticates Hugging Face model downloads; it is **not** a paid hosted inference call. Many specialized models are not deployed by a hosted provider. No GPU is required, but CPU inference and first downloads can be slow. Running every task downloads several GB in total. Prefer running the sections you need; 8 GB RAM may work with sequential cleanup, while 16 GB is more comfortable. Models remain in the disk cache after RAM cleanup.

**LangChain integration:** generation, translation and summarization use `langchain_huggingface.HuggingFacePipeline`; embeddings use `HuggingFaceEmbeddings`. Classification, NER, QA and fill-mask use Transformers pipelines directly. Reranking uses a Sentence Transformers CrossEncoder directly. This keeps the examples simple and avoids `RunnableLambda`.

Target: **Python 3.11.9** in a clean virtual environment or Colab. The examples are text/NLP tasks, not image/audio/video tasks. They are separate from the earlier prompted-chat notebook.

**Validation:** notebook JSON, cell syntax, model uniqueness and example counts checked. Model cards were reviewed. Full downloads and inference were not executed during creation; outputs are not precomputed benchmark results.

## Task → model mapping

| Task | Dedicated model |
|---|---|
| 1. Sentiment analysis | [distilbert/distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) |
| 2. Emotion detection | [bhadresh-savani/distilbert-base-uncased-emotion](https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion) |
| 3. Zero-shot topic classification | [typeform/distilbert-base-uncased-mnli](https://huggingface.co/typeform/distilbert-base-uncased-mnli) |
| 4. Named entity recognition | [dslim/bert-base-NER](https://huggingface.co/dslim/bert-base-NER) |
| 5. Extractive question answering | [distilbert/distilbert-base-cased-distilled-squad](https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad) |
| 6. Summarization | [sshleifer/distilbart-cnn-12-6](https://huggingface.co/sshleifer/distilbart-cnn-12-6) |
| 7. English to French translation | [Helsinki-NLP/opus-mt-en-fr](https://huggingface.co/Helsinki-NLP/opus-mt-en-fr) |
| 8. Fill-mask prediction | [distilbert/distilbert-base-uncased](https://huggingface.co/distilbert/distilbert-base-uncased) |
| 9. Text continuation | [distilbert/distilgpt2](https://huggingface.co/distilbert/distilgpt2) |
| 10. Grammar correction | [vennify/t5-base-grammar-correction](https://huggingface.co/vennify/t5-base-grammar-correction) |
| 11. Instruction-based answering | [google/flan-t5-small](https://huggingface.co/google/flan-t5-small) |
| 12. Toxicity classification | [unitary/toxic-bert](https://huggingface.co/unitary/toxic-bert) |
| 13. Language identification | [papluca/xlm-roberta-base-language-detection](https://huggingface.co/papluca/xlm-roberta-base-language-detection) |
| 14. Financial sentiment | [ProsusAI/finbert](https://huggingface.co/ProsusAI/finbert) |
| 15. SMS spam detection | [mrm8488/bert-tiny-finetuned-sms-spam-detection](https://huggingface.co/mrm8488/bert-tiny-finetuned-sms-spam-detection) |
| 16. News category classification | [dima806/news-category-classifier-distilbert](https://huggingface.co/dima806/news-category-classifier-distilbert) |
| 17. Customer intent classification | [Serj/intent-classifier](https://huggingface.co/Serj/intent-classifier) |
| 18. Sentence similarity | [sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) |
| 19. Search result reranking | [cross-encoder/ms-marco-MiniLM-L6-v2](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2) |
| 20. Answer-aware question generation | [valhalla/t5-small-qg-hl](https://huggingface.co/valhalla/t5-small-qg-hl) |
| 21. English to German translation | [Helsinki-NLP/opus-mt-en-de](https://huggingface.co/Helsinki-NLP/opus-mt-en-de) |
| 22. Paraphrase generation | [Vamsi/T5_Paraphrase_Paws](https://huggingface.co/Vamsi/T5_Paraphrase_Paws) |

## Setup 1 — install
Run once and restart the kernel before continuing. Transformers is pinned to the v4 series because these examples use its summarization/translation/text2text pipelines. This is a constrained environment, not a fully locked dependency set.

In [ ]:
%pip install -q "torch>=2.6,<3" "transformers==4.57.6" "langchain-huggingface==0.3.1" "langchain-core>=0.3.70,<1" "huggingface-hub>=0.34,<1" "sentence-transformers>=3.4,<4" "python-dotenv>=1,<2" "sentencepiece>=0.2,<0.3" "protobuf>=4,<7" "sacremoses>=0.1,<0.2" "numpy>=1.26,<3"

## Setup 2 — `.env`
Create `.env` in the working folder, with:
```dotenv
HUGGINGFACE_API_KEY=hf_replace_with_your_token
```
Create a read token at https://huggingface.co/settings/tokens. In Colab, upload `.env` through the Files panel. This notebook uses `load_dotenv()` (not `load_env()`).

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not HF_TOKEN or HF_TOKEN == "hf_replace_with_your_token":
    raise ValueError("Add your Hugging Face token to .env and rerun this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
print("Token loaded; value hidden.")

## Setup 3 — imports
Use `pipe(input_data)` to run the Transformers pipeline directly. Use `reranker.predict(pairs)` for reranking.

In [ ]:
import gc
import torch
import numpy as np
from transformers import pipeline, set_seed
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

set_seed(42)
print("Device: CPU")

## Task 1 — Sentiment analysis

**Model:** [distilbert/distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english)

Binary POSITIVE/NEGATIVE sentiment; no neutral class.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
pipe = pipeline('text-classification', model=model_id, token=HF_TOKEN, device=-1)
print("Label mapping:", pipe.model.config.id2label)

### Example 1

In [ ]:
input_data = 'The service was excellent.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'The app is terrible and keeps crashing.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 2 — Emotion detection

**Model:** [bhadresh-savani/distilbert-base-uncased-emotion](https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion)

Returns a learned emotion label and score.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'bhadresh-savani/distilbert-base-uncased-emotion'
pipe = pipeline('text-classification', model=model_id, token=HF_TOKEN, device=-1)
print("Label mapping:", pipe.model.config.id2label)

### Example 1

In [ ]:
input_data = 'I am so happy to see my family!'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'I am afraid of losing my job.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 3 — Zero-shot topic classification

**Model:** [typeform/distilbert-base-uncased-mnli](https://huggingface.co/typeform/distilbert-base-uncased-mnli)

Candidate labels are supplied at inference time.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'typeform/distilbert-base-uncased-mnli'
pipe = pipeline('zero-shot-classification', model=model_id, token=HF_TOKEN, device=-1)
task_chain = RunnableLambda(pipe)

### Example 1

In [ ]:
text = 'The customer needs a new debit card.'
# LangChain invoke kwargs are forwarded to the pipeline.
result = pipe(text, candidate_labels=["banking", "healthcare", "education"])
print(result)

### Example 2

In [ ]:
text = 'The patient has an appointment with a doctor.'
# LangChain invoke kwargs are forwarded to the pipeline.
result = pipe(text, candidate_labels=["banking", "healthcare", "education"])
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 4 — Named entity recognition

**Model:** [dslim/bert-base-NER](https://huggingface.co/dslim/bert-base-NER)

Recognizes PER, ORG, LOC and MISC; dates are not a dedicated entity class.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'dslim/bert-base-NER'
pipe = pipeline('token-classification', model=model_id, token=HF_TOKEN, device=-1, aggregation_strategy='simple')
task_chain = RunnableLambda(pipe)

### Example 1

In [ ]:
input_data = 'Anita works at Microsoft in London.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'Rahul visited Paris with John.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 5 — Extractive question answering

**Model:** [distilbert/distilbert-base-cased-distilled-squad](https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad)

Selects an answer span from context. This SQuAD model is not a reliable unanswerable-question detector.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'distilbert/distilbert-base-cased-distilled-squad'
pipe = pipeline('question-answering', model=model_id, token=HF_TOKEN, device=-1)
task_chain = RunnableLambda(pipe)

### Example 1

In [ ]:
input_data = {'question': 'When does the branch close?', 'context': 'The branch opens at 9 AM and closes at 4 PM.'}
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = {'question': 'Where does Anita work?', 'context': 'Anita works at IBM in Hyderabad.'}
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 6 — Summarization

**Model:** [sshleifer/distilbart-cnn-12-6](https://huggingface.co/sshleifer/distilbart-cnn-12-6)

This is one of the larger models in the notebook (roughly 300M parameters). CPU inference can be slow.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'sshleifer/distilbart-cnn-12-6'
pipe = pipeline(
    'summarization', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=60, min_length=10, do_sample=False
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'The city bank introduced a new mobile application on Monday. Customers can now block lost cards instantly, view recent transactions and download monthly statements. The bank said these features were added after customers asked for faster self-service options. Branch services and existing account terms remain unchanged. Staff will help customers learn the new application during the next two weeks.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'The cloud operations team reviewed service performance after a busy weekend. Demand rose sharply during an online sale, causing slow responses for several customers. Engineers increased server capacity and restored normal response times. The team will now introduce workload forecasting and earlier alerts. Managers approved the changes and requested a review next month.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 7 — English to French translation

**Model:** [Helsinki-NLP/opus-mt-en-fr](https://huggingface.co/Helsinki-NLP/opus-mt-en-fr)

A dedicated English–French translation model.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'Helsinki-NLP/opus-mt-en-fr'
pipe = pipeline(
    'translation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=80
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'The customer wants to open a savings account.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'Please send the statement tomorrow.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 8 — Fill-mask prediction

**Model:** [distilbert/distilbert-base-uncased](https://huggingface.co/distilbert/distilbert-base-uncased)

Uses a masked-language-model head; returns candidate tokens.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'distilbert/distilbert-base-uncased'
pipe = pipeline('fill-mask', model=model_id, token=HF_TOKEN, device=-1, top_k=3)
task_chain = RunnableLambda(pipe)

### Example 1

In [ ]:
input_data = 'Paris is the [MASK] of France.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'The doctor works in a [MASK].'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 9 — Text continuation

**Model:** [distilbert/distilgpt2](https://huggingface.co/distilbert/distilgpt2)

A small base language model, not an instruction-following chatbot; continuations may be repetitive or inaccurate.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'distilbert/distilgpt2'
pipe = pipeline(
    'text-generation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=40, do_sample=False, return_full_text=False, pad_token_id=50256
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'Once upon a time in a small village,'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'The future of cloud computing is'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 10 — Grammar correction

**Model:** [vennify/t5-base-grammar-correction](https://huggingface.co/vennify/t5-base-grammar-correction)

Uses the grammar: prefix expected by this fine-tuned model.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'vennify/t5-base-grammar-correction'
pipe = pipeline(
    'text2text-generation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=60, do_sample=False
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'grammar: She go to school every day.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'grammar: He have two bank accounts.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 11 — Instruction-based answering

**Model:** [google/flan-t5-small](https://huggingface.co/google/flan-t5-small)

A general instruction-tuned model assigned to this task; it is not a factual knowledge guarantee.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'google/flan-t5-small'
pipe = pipeline(
    'text2text-generation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=80, do_sample=False
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'Answer briefly: What is a savings account?'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'Answer briefly: What is cloud computing?'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 12 — Toxicity classification

**Model:** [unitary/toxic-bert](https://huggingface.co/unitary/toxic-bert)

Multi-label toxicity scores; categories can overlap. Scores are not a production safety guarantee.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'unitary/toxic-bert'
pipe = pipeline('text-classification', model=model_id, token=HF_TOKEN, device=-1, top_k=None, function_to_apply='sigmoid')
print("Label mapping:", pipe.model.config.id2label)

### Example 1

In [ ]:
input_data = 'You are an idiot and completely useless.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'Thank you for your helpful explanation.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 13 — Language identification

**Model:** [papluca/xlm-roberta-base-language-detection](https://huggingface.co/papluca/xlm-roberta-base-language-detection)

Returns a language code such as fr or es. This multilingual model is larger than the DistilBERT models.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'papluca/xlm-roberta-base-language-detection'
pipe = pipeline('text-classification', model=model_id, token=HF_TOKEN, device=-1)
print("Label mapping:", pipe.model.config.id2label)

### Example 1

In [ ]:
input_data = 'Bonjour, comment allez-vous?'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'Hola, necesito ayuda con mi cuenta.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 14 — Financial sentiment

**Model:** [ProsusAI/finbert](https://huggingface.co/ProsusAI/finbert)

Finance-specific positive/negative/neutral sentiment, distinct from the general sentiment model.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'ProsusAI/finbert'
pipe = pipeline('text-classification', model=model_id, token=HF_TOKEN, device=-1)
print("Label mapping:", pipe.model.config.id2label)

### Example 1

In [ ]:
input_data = 'The company reported record profits and strong revenue growth.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'The company reported a large loss and declining sales.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 15 — SMS spam detection

**Model:** [mrm8488/bert-tiny-finetuned-sms-spam-detection](https://huggingface.co/mrm8488/bert-tiny-finetuned-sms-spam-detection)

Inspect id2label below. The training classes are 0 = ham, 1 = spam; some configs show generic LABEL_0/LABEL_1.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'mrm8488/bert-tiny-finetuned-sms-spam-detection'
pipe = pipeline('text-classification', model=model_id, token=HF_TOKEN, device=-1)
print("Label mapping:", pipe.model.config.id2label)

### Example 1

In [ ]:
input_data = 'Congratulations! You won a free prize. Call now to claim your reward!'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'Hi, are we still meeting for lunch tomorrow?'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 16 — News category classification

**Model:** [dima806/news-category-classifier-distilbert](https://huggingface.co/dima806/news-category-classifier-distilbert)

Uses fixed news categories learned during training; inspect the label mapping.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'dima806/news-category-classifier-distilbert'
pipe = pipeline('text-classification', model=model_id, token=HF_TOKEN, device=-1)
print("Label mapping:", pipe.model.config.id2label)

### Example 1

In [ ]:
input_data = 'The football team won the championship after a dramatic final.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'The government announced a new education policy for public schools.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 17 — Customer intent classification

**Model:** [Serj/intent-classifier](https://huggingface.co/Serj/intent-classifier)

This dedicated T5 intent model uses its model-card prompt format and supplied options.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'Serj/intent-classifier'
pipe = pipeline(
    'text2text-generation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=25, do_sample=False
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = "Topic %% Customer: I want to cancel my subscription. END MESSAGE OPTIONS: cancel subscription % refund % damaged item Choose one topic that matches customer's issue. Class name:"
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = "Topic %% Customer: I want my money back for this purchase. END MESSAGE OPTIONS: cancel subscription % refund % damaged item Choose one topic that matches customer's issue. Class name:"
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 18 — Sentence similarity

**Model:** [sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

Uses HuggingFaceEmbeddings locally. Cosine similarity is not a probability or an accuracy metric.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = HuggingFaceEmbeddings(
    model_name=model_id,
    model_kwargs={"device": "cpu", "token": HF_TOKEN},
    encode_kwargs={"normalize_embeddings": True},
)

### Example 1

In [ ]:
sentences = ['I forgot my password.', 'I need to reset my password.']
vectors = np.asarray(embedding_model.embed_documents(sentences))
print("Vector shape:", vectors.shape)
print("Cosine similarity:", round(float(vectors[0] @ vectors[1]), 4))

### Example 2

In [ ]:
sentences = ['I lost my debit card.', 'The weather is sunny today.']
vectors = np.asarray(embedding_model.embed_documents(sentences))
print("Vector shape:", vectors.shape)
print("Cosine similarity:", round(float(vectors[0] @ vectors[1]), 4))

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del embedding_model
gc.collect()

## Task 19 — Search result reranking

**Model:** [cross-encoder/ms-marco-MiniLM-L6-v2](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2)

A cross-encoder jointly scores each query–document pair. Scores are raw relevance logits, not cosine similarities.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'cross-encoder/ms-marco-MiniLM-L6-v2'
reranker = CrossEncoder(model_id, device="cpu", automodel_args={"token": HF_TOKEN})
task_chain = RunnableLambda(reranker.predict)

### Example 1

In [ ]:
pairs = [['How can I reset my password?', 'Use the forgot-password option on the login page.'], ['How can I reset my password?', 'Debit cards arrive within seven days.']]
scores = pipe(pairs)
for index in np.argsort(scores)[::-1]:
    print(round(float(scores[index]), 4), pairs[index][1])

### Example 2

In [ ]:
pairs = [['How does cloud scaling work?', 'The platform adds servers when workload increases.'], ['How does cloud scaling work?', 'The company changed its office address.']]
scores = pipe(pairs)
for index in np.argsort(scores)[::-1]:
    print(round(float(scores[index]), 4), pairs[index][1])

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del reranker
gc.collect()

## Task 20 — Answer-aware question generation

**Model:** [valhalla/t5-small-qg-hl](https://huggingface.co/valhalla/t5-small-qg-hl)

The answer span is highlighted with <hl> tokens; review the generated question for answerability.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'valhalla/t5-small-qg-hl'
pipe = pipeline(
    'text2text-generation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=60, do_sample=False
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'generate question: The branch closes at <hl> 4 PM <hl> every weekday. </s>'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'generate question: Anita works at <hl> IBM <hl> in Hyderabad. </s>'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 21 — English to German translation

**Model:** [Helsinki-NLP/opus-mt-en-de](https://huggingface.co/Helsinki-NLP/opus-mt-en-de)

A different translation model from English–French.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'Helsinki-NLP/opus-mt-en-de'
pipe = pipeline(
    'translation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=80
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'The bank is closed on Sunday.'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'I would like to check my account balance.'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Task 22 — Paraphrase generation

**Model:** [Vamsi/T5_Paraphrase_Paws](https://huggingface.co/Vamsi/T5_Paraphrase_Paws)

Fine-tuned on PAWS; paraphrases can copy the source or change meaning, so review them.

Run this section in order. After cleanup, rerun the model-loading cell before repeating an example.

In [ ]:
model_id = 'Vamsi/T5_Paraphrase_Paws'
pipe = pipeline(
    'text2text-generation', model=model_id, token=HF_TOKEN, device=-1,
    max_new_tokens=80, do_sample=True, top_p=0.95, temperature=0.8
)
task_chain = HuggingFacePipeline(pipeline=pipe)

### Example 1

In [ ]:
input_data = 'paraphrase: Please send me the account statement. </s>'
result = pipe(input_data)
print(result)

### Example 2

In [ ]:
input_data = 'paraphrase: The application is unavailable during maintenance. </s>'
result = pipe(input_data)
print(result)

**Release model from RAM** (downloaded files stay cached on disk).

In [ ]:
del pipe
gc.collect()

## Troubleshooting and interpretation

- **Import errors:** use a clean Python 3.11 environment, install the listed versions, then restart the kernel. Avoid running these v4 pipeline examples under Transformers 5.
- **401/403 during download:** check the token, read permissions and model access. These examples do not call hosted Inference Providers.
- **Slow downloads:** each section uses a different model. Files are cached; later runs usually reuse them.
- **RAM pressure:** run cleanup after each section or restart the kernel and run only one task. Do not remove the cleanup cells if executing the whole notebook.
- **Model outputs:** scores indicate model confidence or relevance, not measured accuracy. Multi-label categories can overlap. Fixed classifiers only recognize their trained labels.
- **Unexpected predictions or generated text:** review against known labels/references. These are two teaching examples per task, not an evaluation dataset.
- **Missing sentencepiece/sacremoses:** rerun installation and restart the kernel.

## Official documentation
- [LangChain HuggingFacePipeline supported tasks](https://reference.langchain.com/python/langchain-huggingface/llms/huggingface_pipeline/HuggingFacePipeline)
- [Transformers v4 pipelines](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)
- [LangChain Hugging Face embeddings](https://docs.langchain.com/oss/python/integrations/text_embedding/sentence_transformers)

Every task above links to its own model card for training scope, limitations and license details.
